# Day 4 — NumPy and pandas
## Master's Course: Quantitative Skills for Biology

Today we move from pure Python to the two libraries that power almost all biological data analysis:

- **NumPy** — fast numerical arrays and mathematical operations
- **pandas** — labelled tabular data (think: Excel, but in Python)

By the end of this session you will be able to:
- Create and manipulate numerical arrays with NumPy
- Load, inspect, filter, and summarise tabular data with pandas
- Apply log2 transformation to expression data
- Compute fold-changes and identify differentially expressed genes

In [ ]:
# Import the libraries — run this cell first!
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)

---
## 1. NumPy arrays

A NumPy **array** is like a Python list, but:
- All elements must be the same type (usually `float64`)
- Mathematical operations work **element-wise** without loops
- Much faster than plain Python lists for numerical work

In [ ]:
# Creating arrays
expr = np.array([5.2, 8.1, 3.4, 11.0, 7.6, 2.1, 9.3, 6.8])
print("Array:", expr)
print("Type:", type(expr))
print("Shape:", expr.shape)   # (8,) means 8 elements, 1 dimension
print("dtype:", expr.dtype)   # float64 by default

In [ ]:
# Element-wise arithmetic — no loop needed!
print("+ 1:",     expr + 1)
print("* 2:",     expr * 2)
print("log2:",    np.log2(expr))      # log2 transform — very common in genomics
print("z-score:", (expr - expr.mean()) / expr.std())

In [ ]:
# Summary statistics
print(f"Mean:    {expr.mean():.2f}")
print(f"Std dev: {expr.std():.2f}")
print(f"Min:     {expr.min():.2f}")
print(f"Max:     {expr.max():.2f}")
print(f"Median:  {np.median(expr):.2f}")

In [ ]:
# Boolean indexing — select elements meeting a condition
high_expr = expr[expr > 7.0]
print("Highly expressed values:", high_expr)

# Find positions (indices) of high-expression genes
indices = np.where(expr > 7.0)[0]
print("At positions:", indices)

### Speed comparison
To motivate NumPy, let's see how much faster it is than plain Python for summing 1 million numbers:

In [ ]:
import time

n = 1_000_000
python_list = list(range(n))
numpy_array = np.arange(n, dtype=float)

# Python list
t0 = time.time()
total = sum(python_list)
t1 = time.time()
print(f"Python list sum:  {(t1-t0)*1000:.2f} ms")

# NumPy array
t0 = time.time()
total = numpy_array.sum()
t1 = time.time()
print(f"NumPy array sum:  {(t1-t0)*1000:.2f} ms")

print("\nNumPy is typically 10-100x faster for numerical operations.")

---
## 2. Generating synthetic expression data

We'll create a realistic-looking RNA-seq dataset to practise with throughout Days 4 and 5.

- **20 genes** (GENE_001 – GENE_020)
- **6 samples**: 3 controls (`ctrl_1/2/3`) and 3 treated (`treat_1/2/3`)
- The first 5 genes are upregulated in treated samples

In [ ]:
np.random.seed(42)   # for reproducibility

n_genes = 20
n_ctrl  = 3
n_treat = 3

gene_names  = [f"GENE_{i:03d}" for i in range(1, n_genes + 1)]
sample_names = [f"ctrl_{i}" for i in range(1, n_ctrl + 1)] + \
               [f"treat_{i}" for i in range(1, n_treat + 1)]

# Generate expression values (log2 scale, normally distributed)
# Control: mean = 8, std = 1 for all genes
ctrl_data  = np.random.normal(loc=8.0, scale=1.0, size=(n_genes, n_ctrl))
# Treated: same for most genes; first 5 genes have +2 log2 units (4-fold up)
treat_mean = np.full(n_genes, 8.0)
treat_mean[:5] = 10.0   # genes 1-5 are upregulated
treat_data = np.random.normal(loc=treat_mean[:, np.newaxis],
                               scale=1.0, size=(n_genes, n_treat))

# Combine into one array
data = np.hstack([ctrl_data, treat_data])

# Create the DataFrame
df = pd.DataFrame(data, index=gene_names, columns=sample_names)
df.index.name = "gene"

print("Shape:", df.shape)
df.head()

---
## 3. Inspecting a DataFrame

The first thing to do with any new dataset is to explore its structure.

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nIndex (genes):", df.index.tolist())
print("\nData types:")
print(df.dtypes)

In [ ]:
# Statistical summary of all columns
df.describe().round(2)

---
## 4. Accessing data

pandas offers two primary ways to select data:
- `.loc[]` — select by **label** (row/column name)
- `.iloc[]` — select by **integer position**

In [ ]:
# Select a single gene (row) by name
print(df.loc["GENE_001"])
print()
# Select a single sample (column)
print(df["ctrl_1"].head())

In [ ]:
# Select multiple columns
ctrl_df = df[["ctrl_1", "ctrl_2", "ctrl_3"]]
print("Control samples:")
print(ctrl_df.head())

# Select by position
print("\nFirst 3 rows, first 2 columns:")
print(df.iloc[:3, :2])

---
## 5. Filtering rows

We often want to select genes that meet a criterion — e.g., genes with high mean expression.

In [ ]:
# Compute mean expression across all samples
df["mean_all"] = df[sample_names].mean(axis=1)  # axis=1 means across columns

# Boolean mask — which genes have mean > 8.5?
high_expr_mask = df["mean_all"] > 8.5
print("Genes with mean > 8.5:")
print(df[high_expr_mask][["mean_all"]])

In [ ]:
# Filter by gene name (index)
print(df.loc["GENE_001":"GENE_005", sample_names])

---
## 6. Adding computed columns — fold-change

A key step in differential expression analysis is computing the **fold-change** between conditions.

In [ ]:
ctrl_cols  = ["ctrl_1",  "ctrl_2",  "ctrl_3"]
treat_cols = ["treat_1", "treat_2", "treat_3"]

# Mean per condition
df["mean_ctrl"]  = df[ctrl_cols].mean(axis=1)
df["mean_treat"] = df[treat_cols].mean(axis=1)

# Log2 fold-change: treated vs control
# We add a small pseudocount (0.01) to avoid log(0)
df["log2fc"] = np.log2((df["mean_treat"] + 0.01) / (df["mean_ctrl"] + 0.01))

print("Summary of fold-changes:")
print(df[["mean_ctrl", "mean_treat", "log2fc"]].round(2))

---
## 7. Sorting

Sort by any column to find the most/least expressed or most changed genes.

In [ ]:
# Top 5 genes by log2 fold-change (upregulated)
top_up = df.sort_values("log2fc", ascending=False).head(5)
print("Top 5 upregulated genes:")
print(top_up[["mean_ctrl", "mean_treat", "log2fc"]].round(2))

---
## 8. Grouping and aggregating

`groupby()` lets you summarise data by a category — e.g., mean expression per condition.

In [ ]:
# Melt the DataFrame to long format for groupby
# Wide: each sample is a column
# Long: one row per (gene, sample) pair
df_long = df[sample_names].reset_index().melt(
    id_vars="gene",
    var_name="sample",
    value_name="expression"
)

# Add condition column
df_long["condition"] = df_long["sample"].apply(
    lambda s: "control" if s.startswith("ctrl") else "treated"
)

print(df_long.head(8))
print("\nShape:", df_long.shape)

In [ ]:
# Mean expression per gene per condition
mean_by_condition = df_long.groupby(["gene", "condition"])["expression"].mean().unstack()
print(mean_by_condition.round(2).head(10))

---
## Exercises

### Exercise 1
Filter the DataFrame to find genes that are **upregulated** (log2fc > 1) **and** have mean control expression above 8.

Print the gene names and their log2fc values.

In [ ]:
# Exercise 1 — your code here


### Exercise 2
Add a column `'variance'` to the DataFrame containing the variance of expression across all 6 samples for each gene.

Sort by variance (descending) and print the top 5 most variable genes. Are these the genes that were upregulated?

In [ ]:
# Exercise 2 — your code here
# Hint: df[sample_names].var(axis=1) computes variance across columns


### Exercise 3
Compute the Pearson correlation between `ctrl_1` and `ctrl_2`.

Are biological replicates well correlated? Try also comparing `ctrl_1` vs `treat_1`.

*Hint: `df['ctrl_1'].corr(df['ctrl_2'])` or `np.corrcoef(a, b)[0,1]`*

In [ ]:
# Exercise 3 — your code here


---
## Summary

| Task | pandas / NumPy |
|------|----------------|
| Load data | `pd.read_csv()` |
| Inspect | `.head()`, `.describe()`, `.shape` |
| Select rows/cols | `.loc[]`, `.iloc[]`, boolean mask |
| Add a column | `df['new'] = ...` |
| Sort | `.sort_values('col')` |
| Group | `.groupby('col').mean()` |
| Log2 transform | `np.log2(array)` |

**Next session:** Matplotlib and seaborn — visualising the expression data we built today.